### PREPROCESSING

In [1]:
import pandas as pd 
import numpy as np 

In [2]:
file_path = "paysim.csv"

df = pd.read_csv(file_path)
print(df.shape)

(6362620, 11)


In [3]:
print(df.columns.tolist())

['step', 'type', 'amount', 'nameOrig', 'oldbalanceOrg', 'newbalanceOrig', 'nameDest', 'oldbalanceDest', 'newbalanceDest', 'isFraud', 'isFlaggedFraud']


In [4]:
print(df["isFraud"].value_counts())

isFraud
0    6354407
1       8213
Name: count, dtype: int64


In [5]:
print(df["step"].min(), "to" , df["step"].max())

1 to 743


In [6]:
# Calculate Time-Based Split Boundaries

In [7]:
min_step = df["step"].min()
max_step = df["step"].max()

total_steps = max_step - min_step + 1
print("Minimum steps:", min_step)
print("Maximum steps:", max_step)

Minimum steps: 1
Maximum steps: 743


In [8]:
train_steps = int(total_steps * 0.70)
val_steps = int(total_steps * 0.15)

train_end = min_step + train_steps - 1
val_end  = train_end + val_steps

print(f" Train: {min_step} to  {train_end}")
print(f"Validation : {train_end + 1} to {val_end} ")
print(f"Test: {val_end + 1} to {max_step}")

 Train: 1 to  520
Validation : 521 to 631 
Test: 632 to 743


### Conclusiion: 
  - Instead of using a random split, we divided the dataset strictly by 
    - transaction time (step) to better reflect real‑world fraud detection.
        - Train (Steps 1–520): Past transactions.
            - The model learns fraud patterns from this historical data.
        - Validation (Steps 521–631): 
            - Future transactions immediately after training.
            - Used to tune thresholds, features,                                                                                                                     and hyperparameters on unseen data
        - Test (Steps 632–743): 
            - The most future transactions.
            - Kept untouched until final evaluation to measure how the model 
            - performs on genuinely unseen future data.

- Note: 
    - We used a time‑based split rather than a random split. 
       - Training was done on past transactions (Steps 1–520), 
    - validation on the next period (Steps 521–631), 
        - and testing on the most future period (Steps 632–743).
    - This approach prevents data leakage and better simulates real‑world 
        - fraud detection, where future transactions are unknown at training time.

In [9]:
# Create Time-Based Train / Validation / Test Sets

In [10]:
df_train = df[(df["step"] >= 1) & (df["step"] <= 520)].copy()
df_val = df[(df["step"] >= 521) & (df["step"] <= 631)].copy()
df_test = df[(df["step"] >= 632) & (df["step"] <= 743)].copy()

print("Train:", df_train.shape)
print("Validation:", df_val.shape)
print("Test:", df_test.shape)

Train: (6082007, 11)
Validation: (191147, 11)
Test: (89466, 11)


In [11]:
print("\n ========  STEP RANGES ================== : \n")
print("Train:", df_train["step"].min(), "to", df_train["step"].max())
print("Validation:", df_val["step"].min(), "to", df_val["step"].max())
print("Test:", df_test["step"].min(), "to", df_test["step"].max())


 ========  STEP RANGES ================== : 

Train: 1 to 520
Validation: 521 to 631
Test: 632 to 743


In [12]:
print("\n ========  FRAUD COUNTS ================== : \n")
print("\n----- TRAIN ------\n")
print(df_train["isFraud"].value_counts())

print("\n----- VALIDATION ------\n")
print(df_val["isFraud"].value_counts())

print("\n----- TEST ------\n")
print(df_test["isFraud"].value_counts())


 ========  FRAUD COUNTS ================== : 


----- TRAIN ------

isFraud
0    6076226
1       5781
Name: count, dtype: int64

----- VALIDATION ------

isFraud
0    189967
1      1180
Name: count, dtype: int64

----- TEST ------

isFraud
0    88214
1     1252
Name: count, dtype: int64


In [13]:
print("\n======== Fraud_Rates==============\n")
print("Train:", df_train["isFraud"].mean() * 100, "%")
print("Validation:", df_val["isFraud"].mean() * 100, "%")
print("Test:", df_test["isFraud"].mean() * 100, "%")


======== Fraud_Rates==============

Train: 0.09505086067806236 %
Validation: 0.6173259323975788 %
Test: 1.3994143026401091 %


### Conclusion: 
    - Train: steps 1–520 → 6,082,007 transactions
    - Validation: steps 521–631 → 191,147 transactions
    - Test: steps 632–743 → 89,466 transactions
    - There is no time overlap between the sets.
    - Fraud prevalence increases substantially over time:
        - Train: 0.095%
        - Validation: 0.617%
        - Test: 1.399%


- Note:
    - The dataset was divided chronologically into training, validation, 
        - and test sets using the `step` feature to preserve the temporal 
        - nature of transactions. The training set contains steps 1–520, 
        - the validation set contains steps 521–631, and the test set contains 
        - steps 632–743, with no overlap between the time periods.


    - The fraud rate increases from 0.095% in the training period to 0.617% in 
        - validation and 1.399% in the test period. 
        - This indicates a significant temporal shift in fraud prevalence. 
        - Therefore, model evaluation should account for this distribution 
        - change rather than relying on a random train-test split, 
        - which could provide an overly optimistic estimate of real-world 
        performance.

In [14]:
#Split Integrity Check
    # After creating the chronological Train, Validation, and Test datasets, 
    # we verify that the split was performed correctly.

#Checks:
    # 1. Total row count: 
          # ---> Ensure no rows were lost or added during splitting.
    # 2. Train before Validation: 
        # ---->Ensure the training period ends before the validation period begins.
    # 3. Validation before Test:
        # ---->Ensure the validation period ends before the test period begins.
    # 4. Correct step ranges:
        # --->Confirm that Train covers steps 1–520,
            # Validation covers 521–631, and Test covers 632–743.

In [15]:
print("\n ------- Total_rows_check ------------- \n")
total_split_rows = len(df_train) + len(df_val) + len(df_test)

print("Original dataset rows :", len(df))
print("Total split rows      :", total_split_rows)

if total_split_rows == len(df):
    print("Row count: Passed")
else:
    print("Row count: Failed")


 ------- Total_rows_check ------------- 

Original dataset rows : 6362620
Total split rows      : 6362620
Row count: Passed


In [16]:
print("\n------- Validation → Train check -------\n")
print("Train_last_ step    : ", df_train["step"].max())
print("Validation_ first_ step :", df_val["step"].min())

if df_train["step"].max() < df_val["step"].min():
    print("Train --> Validation: Passed")
else:
    print("Train --> Validation: Failed")


------- Validation → Train check -------

Train_last_ step    :  520
Validation_ first_ step : 521
Train --> Validation: Passed


In [17]:
print("\n------- Validation → Test check -------\n")

print("Validation last step :", df_val["step"].max())
print("Test first step     :", df_test["step"].min())

if df_val["step"].max() < df_test["step"].min():
    print("Validation → Test: PASSED")
else:
    print("Validation → Test: FAILED")


------- Validation → Test check -------

Validation last step : 631
Test first step     : 632
Validation → Test: PASSED


In [18]:
print("\n---------- Step Ranges -------------\n")
print("Train    :", df_train["step"].min(), "to", df_train["step"].max())
print("Validation  :", df_val["step"].min(), "to", df_val["step"].max())
print("Test    :", df_test["step"].min(), "to", df_test["step"].max())


if (
    df_train["step"].min() == 1
    and df_train["step"].max() == 520
    and df_val["step"].min() == 521
    and df_val["step"].max() == 631
    and df_test["step"].min() == 632
    and df_test["step"].max() == 743
):
    print("\n Step ranges: Passed")
else:
    print("\n Step ranges : Failed")


---------- Step Ranges -------------

Train    : 1 to 520
Validation  : 521 to 631
Test    : 632 to 743

 Step ranges: Passed


### Conclusion: 
- The chronological data split was successfully verified. 
    - The complete dataset contains 6,362,620 transactions,
    - and the combined Train, Validation,
    - and Test sets contain the same number of transactions,
    - confirming that no rows were lost or added during the split.

- The time periods are also correctly separated:
    - Train covers steps 1–520, Validation covers steps 521–631, 
    - and Test covers steps 632–743. Therefore, 
    - the datasets are temporally ordered with no overlap, 
    - providing a suitable foundation for leakage-aware model development and evaluation.

### ========== Feature Enginneering =============

## Feature Engineering
- Feature engineering is used to transform the raw transaction data 
    - into meaningful variables that can help the machine learning model 
    - identify fraudulent behaviour.
- We will first create transaction-level behavioural features using information
    -available within each transaction. 
    - These features will capture balance inconsistencies, 
    - transaction-to-balance relationships, 
    - and balance depletion patterns without using the target variable.

In [19]:
def create_basic_features(data):
    data = data.copy()

    data["origin_balance_error"] = (
        data["oldbalanceOrg"]
        - data["amount"]
        - data["newbalanceOrig"]
    )

    data["abs_origin_balance_error"] = (
        data["origin_balance_error"].abs()
    )

    data["amount_to_origin_ratio"] = np.where(
        data["oldbalanceOrg"] > 0,
        data["amount"] / data["oldbalanceOrg"],
        np.nan
    )

    data["sender_balance_depleted"] = (
        data["newbalanceOrig"] == 0
    ).astype(int)

    data["destination_balance_depleted"] = (
        data["newbalanceDest"] == 0 
    ).astype(int)

    return data

In [20]:
df_train = create_basic_features(df_train)
df_val = create_basic_features(df_val)
df_test = create_basic_features(df_test)

In [21]:
new_features = [
    "origin_balance_error",
    "abs_origin_balance_error",
    "amount_to_origin_ratio",
    "sender_balance_depleted",
    "destination_balance_depleted"
]

print("New Features Created : .........")
print(new_features)

New Features Created : .........
['origin_balance_error', 'abs_origin_balance_error', 'amount_to_origin_ratio', 'sender_balance_depleted', 'destination_balance_depleted']


In [22]:
print("\n Train Shape: ", df_train.shape)
print("Validation shape: ", df_val.shape)
print("Test Shape: ", df_test.shape)


 Train Shape:  (6082007, 16)
Validation shape:  (191147, 16)
Test Shape:  (89466, 16)


In [23]:
print(df_train[new_features].head())

   origin_balance_error  abs_origin_balance_error  amount_to_origin_ratio  \
0                   0.0                       0.0                0.057834   
1                   0.0                       0.0                0.087735   
2                   0.0                       0.0                1.000000   
3                   0.0                       0.0                1.000000   
4                   0.0                       0.0                0.280795   

   sender_balance_depleted  destination_balance_depleted  
0                        0                             1  
1                        0                             1  
2                        1                             1  
3                        1                             1  
4                        0                             1  


### =============== Feature Validation =============

    - After creating the behavioural features,
    - we validate their data quality before proceeding. 
    - We check for missing values and infinite values because ratios and
    - arithmetic operations can produce invalid values,
    - which may cause problems during preprocessing or model training.

In [24]:
for name, data in [
    ("Train", df_train),
    ("Validation", df_val),
    ("Test",df_test)
]:

    print(f"\n =================== {name.upper()} ================\n ")

    missing_values = data[new_features].isna().sum()

    infinite_values = np.isinf(
        data[new_features].select_dtypes(include = np.number)
    ).sum()

    print("\n ------  MISSING VALUES --------\n", missing_values)
    print("\n ------  INFINITE VALUES  --------\n", infinite_values)

    print("\n------ TOTAL MISSING VALUES ------- \n", missing_values.sum())
    print("\n------- TOTAL INFINITE VALUES --------- \n ", infinite_values.sum())


 =================== TRAIN ================
 

 ------  MISSING VALUES --------
 origin_balance_error                  0
abs_origin_balance_error              0
amount_to_origin_ratio          2023376
sender_balance_depleted               0
destination_balance_depleted          0
dtype: int64

 ------  INFINITE VALUES  --------
 origin_balance_error            0
abs_origin_balance_error        0
amount_to_origin_ratio          0
sender_balance_depleted         0
destination_balance_depleted    0
dtype: int64

------ TOTAL MISSING VALUES ------- 
 2023376

------- TOTAL INFINITE VALUES --------- 
  0

 =================== VALIDATION ================
 

 ------  MISSING VALUES --------
 origin_balance_error                0
abs_origin_balance_error            0
amount_to_origin_ratio          56003
sender_balance_depleted             0
destination_balance_depleted        0
dtype: int64

 ------  INFINITE VALUES  --------
 origin_balance_error            0
abs_origin_balance_error       

In [25]:
# VERIFY RATIO NaN VALUES

In [26]:
for name, data in [
    ("Train", df_train),
    ("Validation", df_val),
    ("Test", df_test)
]:

    nan_rows = data[data["amount_to_origin_ratio"].isna()]

    print(f"\n ================ {name.upper()} ============ ")

    print("NAN ratio Rows: ", len(nan_rows))

    print(
        "Rows with oldbalanceOrg = 0 :",
        (nan_rows["oldbalanceOrg"] == 0).sum()
    )

    print(
        "Rows with oldbalanceOrg == 0 : ",
        (nan_rows["oldbalanceOrg"] == 0).sum()
    )

    print(
        "Rows with oldbalanceOrg > 0 : ",
        (nan_rows["oldbalanceOrg"] > 0).sum()
    )


 ================ TRAIN ============ 
NAN ratio Rows:  2023376
Rows with oldbalanceOrg = 0 : 2023376
Rows with oldbalanceOrg == 0 :  2023376
Rows with oldbalanceOrg > 0 :  0

 ================ VALIDATION ============ 
NAN ratio Rows:  56003
Rows with oldbalanceOrg = 0 : 56003
Rows with oldbalanceOrg == 0 :  56003
Rows with oldbalanceOrg > 0 :  0

 ================ TEST ============ 
NAN ratio Rows:  23070
Rows with oldbalanceOrg = 0 : 23070
Rows with oldbalanceOrg == 0 :  23070
Rows with oldbalanceOrg > 0 :  0


## Analysis of Zero Origin Balance
- The `amount_to_origin_ratio` feature is undefined when the sender's initial
    balance is zero.
- Instead of treating these values as ordinary missing data,
- we examine zero-origin-balance transactions separately because a zero
   starting balance represents a distinct transaction condition.

In [27]:
for name, data in[
    ("Train",df_train),
    ("Validation",df_val),
    ("Test",df_test)
]:

    zero_balance = data["oldbalanceOrg"] == 0

    total_zero_balance = zero_balance.sum()
    fraud_zero_balance = data.loc[zero_balance, "isFraud"].sum()

    fraud_rate = (
        fraud_zero_balance / total_zero_balance * 100
        if total_zero_balance > 0 else 0
    )

    print(f"\n=============={name.upper()}==============\n")

    print("Zero Origin Balance transactions: ", total_zero_balance)
    print("Fraud Transactions : ", fraud_zero_balance)
    print("Fraud Rate : ", fraud_rate, "%")


==============TRAIN==============

Zero Origin Balance transactions:  2023376
Fraud Transactions :  31
Fraud Rate :  0.0015320928982057707 %

==============VALIDATION==============

Zero Origin Balance transactions:  56003
Fraud Transactions :  3
Fraud Rate :  0.005356855882720569 %

==============TEST==============

Zero Origin Balance transactions:  23070
Fraud Transactions :  7
Fraud Rate :  0.03034243606415258 %


## Handling Zero Origin Balance
- Since the ratio is undefined when `oldbalanceOrg` is zero,
- we create a separate binary feature to preserve this information.
- The undefined ratio values are then represented as 0,
- while the new binary feature indicates whether the 
- sender had a zero balance before the transaction.

In [28]:
def handle_zero_origin_balance(data):
    data = data.copy()

    data["origin_balance_zero"] = (
        data["oldbalanceOrg"] == 0
    ).astype(int)

    data["amount_to_origin_ratio"] = (
        data["amount_to_origin_ratio"].fillna(0)
    )

    return data

In [29]:
df_train = handle_zero_origin_balance(df_train)
df_val = handle_zero_origin_balance(df_val)
df_test = handle_zero_origin_balance(df_test)

In [30]:
print("\n****************** Missing Values ****************** \n")
print("Train NaN ratio values : ",
     df_train["amount_to_origin_ratio"].isna().sum())
print("Validation NaN ratio values : ",
     df_val["amount_to_origin_ratio"].isna().sum())
print("Test NaN ratio values : ",
     df_test["amount_to_origin_ratio"].isna().sum())


****************** Missing Values ****************** 

Train NaN ratio values :  0
Validation NaN ratio values :  0
Test NaN ratio values :  0


In [31]:
print("\n****************** Value Counts ******************\n")

print("\n --------- Train : ----------  \n",
     df_train["origin_balance_zero"].value_counts())
print("\n ---------- Validation ----------\n ",
     df_val["origin_balance_zero"].value_counts())
print("\n ---------- Test ----------\n",
     df_test["origin_balance_zero"].value_counts())


****************** Value Counts ******************


 --------- Train : ----------  
 origin_balance_zero
0    4058631
1    2023376
Name: count, dtype: int64

 ---------- Validation ----------
  origin_balance_zero
0    135144
1     56003
Name: count, dtype: int64

 ---------- Test ----------
 origin_balance_zero
0    66396
1    23070
Name: count, dtype: int64


## Time-Based Feature Engineering

The `step` variable represents the hourly progression of the simulated transactions. We derive time-based features from it to help the model capture temporal patterns in transaction behaviour.

The `step` value is converted into:
- `hour_of_day`: the hour within a 24-hour cycle.
- `day_of_week`: the day within a 7-day cycle.
- `is_weekend`: whether the transaction occurs on a weekend.

In [32]:
def create_time_features(data):
    data = data.copy()

    data["hour_of_day"] = (data["step"] - 1) % 24

    data["day_of_week"] = ((data["step"] - 1) // 24) % 7

    data["is_weekend"] = (
        data["day_of_week"] >= 5
    ).astype(int)

    return data

In [33]:
df_train = create_time_features(df_train)
df_val = create_time_features(df_val)
df_test = create_time_features(df_test)

In [34]:
time_features = [
    "hour_of_day",
    "day_of_week",
    "is_weekend"
]

print("\n New time Features ------>", time_features)
print("\n Train shape: ", df_train.shape)
print("\n Validation Shape: ", df_val.shape)
print("\n Test Shape : ", df_test.shape)


 New time Features ------> ['hour_of_day', 'day_of_week', 'is_weekend']

 Train shape:  (6082007, 20)

 Validation Shape:  (191147, 20)

 Test Shape :  (89466, 20)


In [35]:
print("\n Feature Preview: ")
print(df_train[["step"] + time_features].head(10))


 Feature Preview: 
   step  hour_of_day  day_of_week  is_weekend
0     1            0            0           0
1     1            0            0           0
2     1            0            0           0
3     1            0            0           0
4     1            0            0           0
5     1            0            0           0
6     1            0            0           0
7     1            0            0           0
8     1            0            0           0
9     1            0            0           0


In [36]:
print("\n Hour Range: ")
print(df_train["hour_of_day"].min(),
      "to",
     df_train["hour_of_day"].max()
)


 Hour Range: 
0 to 23


In [37]:
print("\n Day-of-week Range: ")
print(df_train["day_of_week"].min(),
     "to",
      df_train["day_of_week"].max()
)


 Day-of-week Range: 
0 to 6


## Balance Behaviour Feature Engineering

- We create additional features to capture how the transaction amount relates to changes in the sender's and receiver's balances. 
- These features help represent transaction behaviour in a form that can be used by machine learning models.

In [38]:
def create_balance_feature(data):
    data = data.copy()

    data["sender_balance_change"] = (
        data["oldbalanceOrg"] - data["newbalanceOrig"]
    )

    data["sender_amount_difference"] = (
        data["amount"] - data["sender_balance_change"]
    )

    data["receiver_balance_change"] = (
        data["newbalanceDest"] - data["oldbalanceDest"]
    )

    data["receiver_amount_difference"] = (
        data["amount"] - data["receiver_balance_change"]
    )

    return data

In [39]:
df_train = create_balance_feature(df_train)
df_val = create_balance_feature(df_val)
df_test = create_balance_feature(df_test)

In [40]:
balance_features = [
    "sender_balance_change",
    "sender_amount_difference",
    "receiver_balance_change",
    "receiver_amount_difference"
]
print("\n New Balance Features : \n", balance_features)


 New Balance Features : 
 ['sender_balance_change', 'sender_amount_difference', 'receiver_balance_change', 'receiver_amount_difference']


In [41]:
print("\n Train Shape: ", df_train.shape)
print("\n Validation Shape: ", df_val.shape)
print("\n Test Shape: ", df_test.shape)


 Train Shape:  (6082007, 24)

 Validation Shape:  (191147, 24)

 Test Shape:  (89466, 24)


In [42]:
print("\n Feature Preview: ",df_train[["amount"]  + balance_features].head())


 Feature Preview:       amount  sender_balance_change  sender_amount_difference  \
0   9839.64                9839.64             -1.455192e-11   
1   1864.28                1864.28              1.136868e-12   
2    181.00                 181.00              0.000000e+00   
3    181.00                 181.00              0.000000e+00   
4  11668.14               11668.14              0.000000e+00   

   receiver_balance_change  receiver_amount_difference  
0                      0.0                     9839.64  
1                      0.0                     1864.28  
2                      0.0                      181.00  
3                 -21182.0                    21363.00  
4                      0.0                    11668.14  


## Validating Balance Behaviour Features
- The newly created balance features are checked for missing values, infinite values, and numerical precision issues. 
- Very small values close to zero may occur due to floating-point arithmetic and should not be interpreted as meaningful transaction differences.

In [43]:
for name,data in[
    ("Train", df_train),
    ("Validation", df_val),
    ("Test", df_test)
]:

    print(f"\n <==****************** {name.upper()} **************************==>")

    print("\n-------- Missing Values ---------\n")
    print(data[balance_features].isna().sum())

    print("\n-------- Infinite Values ---------\n")
    print(
        np.isinf(
            data[balance_features].select_dtypes(include = np.number)
        ).sum()
    )

    print("\n Values Extremely close to zero --------->  \n")

    for feature in [
        "sender_amount_difference",
        "receiver_amount_difference"
    ]:
        count_near_zero = (
            np.abs(data[feature]) < 1e-8
        ).sum()

        print(
            feature,
            ":",
            count_near_zero
        )


 <==****************** TRAIN **************************==>

-------- Missing Values ---------

sender_balance_change         0
sender_amount_difference      0
receiver_balance_change       0
receiver_amount_difference    0
dtype: int64

-------- Infinite Values ---------

sender_balance_change         0
sender_amount_difference      0
receiver_balance_change       0
receiver_amount_difference    0
dtype: int64

 Values Extremely close to zero --------->  

sender_amount_difference : 1174719
receiver_amount_difference : 1904582

 <==****************** VALIDATION **************************==>

-------- Missing Values ---------

sender_balance_change         0
sender_amount_difference      0
receiver_balance_change       0
receiver_amount_difference    0
dtype: int64

-------- Infinite Values ---------

sender_balance_change         0
sender_amount_difference      0
receiver_balance_change       0
receiver_amount_difference    0
dtype: int64

 Values Extremely close to zero --------->  


## Balance Feature Behaviour by Fraud Status

We compare the newly created balance features between fraudulent and non-fraudulent transactions to determine whether they capture meaningful behavioural differences.

A feature should not be retained merely because it can be calculated. Its distribution and relationship with the target should also be examined before selecting it for the final model.

In [44]:
for feature in balance_features:

    print("\n" + "=" * 60)
    print(f"Feature: {feature}")
    print("=" * 60)

    print(
        df_train.groupby("isFraud")[feature].agg(
            ["count", "mean", "median", "std", "min", "max"]
        )
    )


Feature: sender_balance_change
           count          mean     median           std        min  \
isFraud                                                              
0        6076226 -2.307598e+04       0.00  1.062644e+05 -1915267.9   
1           5781  1.450903e+06  443893.69  2.348964e+06        0.0   

                 max  
isFraud               
0         4164236.31  
1        10000000.00  

Feature: sender_amount_difference
           count           mean   median            std           min  \
isFraud                                                                 
0        6076226  202273.465232  69507.0  615125.403828 -1.000000e-02   
1           5781    7151.229327      0.0  208306.347738 -3.725290e-09   

                 max  
isFraud               
0        92445516.64  
1        10000000.00  

Feature: receiver_balance_change
           count           mean  median           std          min  \
isFraud                                                              
0

## Robust Comparison of Balance Features

- Mean values can be strongly affected by extreme transactions.
- Therefore, we compare the distributions using the median and percentiles to determine whether the observed differences between fraudulent and non-fraudulent transactions are consistent or mainly driven by outliers.

In [ ]:
for feature in balance_features:

    print("\n" + "=" * 65)
    print(f"FEATURE: {feature}")
    print("=" * 65)

    result = df_train.groupby("isFraud")[feature].quantile(
        [0.25, 0.50, 0.75, 0.95, 0.99]
    ).unstack()

    result.columns = [
        "Q25",
        "Median",
        "Q75",
        "Q95",
        "Q99"
    ]

    print(result)


FEATURE: sender_balance_change
               Q25     Median         Q75         Q95           Q99
isFraud                                                            
0             0.00       0.00    10085.58    72852.00  2.103714e+05
1        126192.63  443893.69  1533798.48  7596886.83  1.000000e+07

FEATURE: sender_amount_difference
              Q25   Median         Q75        Q95           Q99
isFraud                                                        
0        3119.175  69507.0  250380.075  702776.08  1.572198e+06
1           0.000      0.0       0.000       0.00  0.000000e+00

FEATURE: receiver_balance_change


In [ ]:
df_train.drop(columns=["sender_amount_difference"], inplace=True)
df_val.drop(columns=["sender_amount_difference"], inplace=True)
df_test.drop(columns=["sender_amount_difference"], inplace=True)

print("sender_amount_difference dropped successfully.")

## Categorical Feature Analysis — Transaction Type

The `type` feature represents the category of each transaction, such as `PAYMENT`, `TRANSFER`, `CASH_OUT`, `CASH_IN`, and `DEBIT`.

Before encoding this categorical feature, we verify the categories present in each dataset split. This helps ensure consistent preprocessing across Train, Validation, and Test data.

In [ ]:
for name, data in [
    ("Train", df_train),
    ("Validation", df_val),
    ("Test", df_test)
]:
    
    print(f"\n========== {name.upper()} ==========")
    
    print("Unique transaction types:")
    print(data["type"].unique())
    
    print("\nNumber of unique types:")
    print(data["type"].nunique())
    
    print("\nTransaction type counts:")
    print(data["type"].value_counts())

## One-Hot Encoding of Transaction Type

The `type` feature is categorical, so it cannot be directly used by most machine learning models in its text form.

We convert the five transaction types into binary indicator features using One-Hot Encoding.

The encoding mapping is learned from the training data and then applied to the validation and test sets. This ensures that preprocessing decisions are not learned from future data.

In [ ]:
from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder(
    handle_unknown = "ignore",
    sparse_output = False
)

train_type_encoded = encoder.fit_transform(df_train[["type"]])
val_type_encoded = encoder.transform(df_val[["type"]])
test_type_encoded = encoder.transform(df_test[["type"]])

encoded_columns = encoder.get_feature_names_out(["type"])

train_type_df = pd.DataFrame(
    train_type_encoded,
    columns = encoded_columns,
    index = df_train.index
)

val_type_df = pd.DataFrame(
    val_type_encoded,
    columns = encoded_columns,
    index = df_val.index
)

test_type_df = pd.DataFrame(
    test_type_encoded,
    columns = encoded_columns,
    index = df_test.index
)

df_train = pd.concat([df_train, train_type_df], axis = 1)
df_val = pd.concat([df_val, val_type_df], axis = 1)
df_test = pd.concat([df_test, test_type_df], axis = 1)

df_train.drop(columns = ["type"], inplace = True)
df_val.drop(columns = ["type"], inplace = True)
df_test.drop(columns = ["type"], inplace = True) 

In [ ]:
print("\n Encoded columns: \n", list(encoded_columns))

In [ ]:
print("\n Train shape:", df_train.shape)
print("\n Validation shape:", df_val.shape)
print("\n Test shape:", df_test.shape)

## Save Processed Datasets

The processed Train, Validation, and Test datasets are saved in Parquet format so that they can be reused in the next stages of the project without repeating the preprocessing steps.

Parquet is used because it provides efficient storage and faster loading for large tabular datasets.

In [ ]:
df_train.to_parquet("train_processed.parquet", index=False)
df_val.to_parquet("validation_processed.parquet", index=False)
df_test.to_parquet("test_processed.parquet", index=False)

print("Processed datasets saved successfully.")

print("\nFiles:")
print("1. train_processed.parquet")
print("2. validation_processed.parquet")
print("3. test_processed.parquet")